# 🍎 Challenge: Domain Adaptation via Neural Style Transfer (NST)
**Student:** Anderson David Arenas Gutiérrez  
**Institución:** Universidad Distrital Francisco José de Caldas - Facultad de Ingeniería  
**Course:** Machine Learning  
**Instructor:** Carlos Andrés Sierra, M.Sc.  

---

## 📑 1. Environment Setup and Seed Definition
To ensure the scientific reproducibility required by the challenge paper, we will run the full pipeline using **3 different random seeds (42, 100, 2026)**. At the end, we will evaluate the impact of the synthetic dataset by comparing average metrics.

In [9]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch_directml
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms, models

# Configure Dedicated AMD GPU through DirectML
device = torch_directml.device(1) if torch_directml.is_available() else torch.device('cpu')
print(f"🚀 Compute device configured: {device}")

# Definition of the 3 required seeds
SEEDS = [42, 100, 2026]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    print(f"🌱 Random seed fixed at: {seed}")

BASE_DIR = "C:\\Users\\Anderson\\Documents\\UD\\7mo\\MachineLearning\\Challenges\\challenge-7_5"
DATA_DIR = os.path.join(BASE_DIR, "data")
LOG_DIR = os.path.join(BASE_DIR, "runs")

🚀 Compute device configured: privateuseone:1


## 📊 2. Loading Pipeline and Data Augmentation
We implement robust training and validation transforms. We set `batch_size = 32` according to the search space suggested in the guide.

In [10]:
imsize = 224
batch_size = 32

# Transform for the Robust Experiment (Augmentation Pesado)
transform_robust_train = transforms.Compose([
    transforms.Resize((imsize, imsize)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Fixed Validation Transform
transform_val = transforms.Compose([
    transforms.Resize((imsize, imsize)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

source_train_dir = os.path.join(DATA_DIR, "source_real", "train")
synthetic_train_dir = os.path.join(DATA_DIR, "synthetic_target")
target_val_dir = os.path.join(DATA_DIR, "target_infograph", "test")

dataset_val = datasets.ImageFolder(target_val_dir, transform=transform_val)
loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)

print(f"Real validation set (Infographics) loaded with {len(dataset_val)} images.")

Real validation set (Infographics) loaded with 1166 images.


## 🛠️ 3. Auxiliary Training Functions

In [11]:
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss, corrects, total = 0.0, 0, 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        corrects += torch.sum(preds == labels.data)
        total += inputs.size(0)
    return running_loss / total, (corrects.double() / total).item()

def evaluate_model(model, dataloader, criterion):
    model.eval()
    running_loss, corrects, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)
            total += inputs.size(0)
    return running_loss / total, (corrects.double() / total).item()

def initialize_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model.to(device)

## 🔁 4. Multi-Seed Execution Loop (Stage-wise Training)
We will run the full pipeline for each of the 3 seeds. Logs will be sent to dynamic TensorBoard subfolders so we can compare the average curves of Stage 1 (Source) against Stage 2 (Synthetic Adaptation).

In [12]:
final_results = {}
checkpoints_dir = os.path.join(BASE_DIR, "checkpoints")
os.makedirs(checkpoints_dir, exist_ok=True)

for seed in SEEDS:
    print(f"\n=======================================================")
    print(f"🏃 RUNNING EXPERIMENT WITH SEED {seed}")
    print(f"=======================================================")
    set_seed(seed)
    
    # Force data reload to apply the current seed sampling
    dataset_source = datasets.ImageFolder(source_train_dir, transform=transform_robust_train)
    dataset_synthetic = datasets.ImageFolder(synthetic_train_dir, transform=transform_robust_train)
    
    loader_source = DataLoader(dataset_source, batch_size=batch_size, shuffle=True)
    loader_synthetic = DataLoader(dataset_synthetic, batch_size=batch_size, shuffle=True)
    
    num_classes = len(dataset_source.classes)
    model = inicializar_modelo(num_classes)
    criterion = nn.CrossEntropyLoss()
    
    # Initialize TensorBoard writers per seed
    writer_st1 = SummaryWriter(os.path.join(LOG_DIR, f"seed_{seed}_stage1_source"))
    writer_st2 = SummaryWriter(os.path.join(LOG_DIR, f"seed_{seed}_stage2_synthetic"))
    
    # ---- STAGE 1: Training on Source Domain (Real Photos) ----
    epochs_st1 = 25
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    print("\n🔹 [Stage 1] Training on Real Source...")
    
    acc_final_st1 = 0.0
    for epoch in range(1, epochs_st1 + 1):
        t0 = time.time()
        loss_t, acc_t = train_one_epoch(model, loader_source, criterion, optimizer)
        loss_v, acc_v = evaluate_model(model, loader_val, criterion)
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch [{epoch:02d}/{epochs_st1:02d}] ({time.time()-t0:.1f}s) -> Loss: {loss_t:.4f} | Acc: {acc_t:.4f} || Val Acc: {acc_v:.4f}")
            
        writer_st1.add_scalar("Loss/Train", loss_t, epoch)
        writer_st1.add_scalar("Accuracy/Train", acc_t, epoch)
        writer_st1.add_scalar("Loss/Validation", loss_v, epoch)
        writer_st1.add_scalar("Accuracy/Validation", acc_v, epoch)
        acc_final_st1 = acc_v
        
    print(f"✅ End Stage 1. Accuracy on Real Infographics: {acc_final_st1:.4f}")
    
    if seed == 42:
        path_pt_a = os.path.join(checkpoints_dir, "best_part_A_model.pt")
        torch.save(model.state_dict(), path_pt_a)
        print(f"💾 Saved Part A checkpoint: {path_pt_a}")
    
    # ---- STAGE 2: Domain Adaptation (Synthetic Target with NST) ----
    epochs_st2 = 40
    optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
    print("\n🔹 [Stage 2] Fine-Tuning with Synthetic Images (NST)...")
    
    acc_final_st2 = 0.0
    for epoch in range(1, epochs_st2 + 1):
        t0 = time.time()
        loss_t, acc_t = train_one_epoch(model, loader_synthetic, criterion, optimizer)
        loss_v, acc_v = evaluate_model(model, loader_val, criterion)
        
        if epoch % 5 == 0 or epoch == 1:
            print(f"  Synth Epoch [{epoch:02d}/{epochs_st2:02d}] ({time.time()-t0:.1f}s) -> Loss: {loss_t:.4f} | Acc: {acc_t:.4f} || Val Acc: {acc_v:.4f}")
            
        writer_st2.add_scalar("Loss/Train", loss_t, epoch)
        writer_st2.add_scalar("Accuracy/Train", acc_t, epoch)
        writer_st2.add_scalar("Loss/Validation", loss_v, epoch)
        writer_st2.add_scalar("Accuracy/Validation", acc_v, epoch)
        acc_final_st2 = acc_v
        
    print(f"✅ End Stage 2. Final Adapted Accuracy: {acc_final_st2:.4f}")
    
    if seed == 42:
        path_pt_c = os.path.join(checkpoints_dir, "best_part_C_model.pt")
        torch.save(model.state_dict(), path_pt_c)
        print(f"💾 Saved Part C checkpoint: {path_pt_c}")
    
    final_results[seed] = {"Stage1_Base": acc_final_st1, "Stage2_Adapted": acc_final_st2}
    
    writer_st1.close()
    writer_st2.close()


🏃 RUNNING EXPERIMENT WITH SEED 42
🌱 Random seed fixed at: 42

🔹 [Stage 1] Training on Real Source...
  Epoch [01/25] (6.3s) -> Loss: 1.2564 | Acc: 0.5560 || Val Acc: 0.3696
  Epoch [05/25] (6.6s) -> Loss: 0.0376 | Acc: 0.9928 || Val Acc: 0.4177
  Epoch [10/25] (6.6s) -> Loss: 0.0159 | Acc: 1.0000 || Val Acc: 0.4631
  Epoch [15/25] (6.5s) -> Loss: 0.0122 | Acc: 0.9964 || Val Acc: 0.4434
  Epoch [20/25] (6.7s) -> Loss: 0.0090 | Acc: 1.0000 || Val Acc: 0.4477
  Epoch [25/25] (6.7s) -> Loss: 0.0028 | Acc: 1.0000 || Val Acc: 0.4168
✅ End Stage 1. Accuracy on Real Infographics: 0.4168
💾 Saved Part A checkpoint: C:\Users\Anderson\Documents\UD\7mo\MachineLearning\Challenges\challenge-7_5\checkpoints\best_part_A_model.pt

🔹 [Stage 2] Fine-Tuning with Synthetic Images (NST)...
  Synth Epoch [01/40] (5.7s) -> Loss: 0.0378 | Acc: 0.9831 || Val Acc: 0.4554
  Synth Epoch [05/40] (5.7s) -> Loss: 0.0101 | Acc: 1.0000 || Val Acc: 0.4940
  Synth Epoch [10/40] (5.8s) -> Loss: 0.0111 | Acc: 1.0000 || Val

## 📈 5. Consolidated Statistical Analysis
We calculate means and standard deviations for report support.

In [13]:
accs_st1 = [res["Stage1_Base"] for res in final_results.values()]
accs_st2 = [res["Stage2_Adapted"] for res in final_results.values()]

print("\n=======================================================")
print("📊 FINAL METRICS REPORT (3 SEEDS)")
print("=======================================================")
print(f"Metrics by seed: {final_results}\n")
print(f"➡️ BASELINE (Stage 1): {np.mean(accs_st1)*100:.2f}% ± {np.std(accs_st1)*100:.2f}%")
print(f"➡️ ADAPTED DOMAIN (Stage 2): {np.mean(accs_st2)*100:.2f}% ± {np.std(accs_st2)*100:.2f}%")
print(f"\n🚀 NET GAIN FROM DOMAIN ADAPTATION: +{(np.mean(accs_st2) - np.mean(accs_st1))*100:.2f}%")


📊 FINAL METRICS REPORT (3 SEEDS)
Metrics by seed: {42: {'Stage1_Base': 0.416809618473053, 'Stage2_Adapted': 0.5111492276191711}, 100: {'Stage1_Base': 0.4802744686603546, 'Stage2_Adapted': 0.536878228187561}, 2026: {'Stage1_Base': 0.38078904151916504, 'Stage2_Adapted': 0.4777015447616577}}

➡️ BASELINE (Stage 1): 42.60% ± 4.11%
➡️ ADAPTED DOMAIN (Stage 2): 50.86% ± 2.42%

🚀 NET GAIN FROM DOMAIN ADAPTATION: +8.26%
